# Evaluation result figures

This notebook summarizes `outputs/eval` into publication-oriented figures. It covers MATH from baseline through Self-Play and the ScienceQA Nature Science subset comparison between PEFT-LoRA and Self-Play. All figures are exported to `docs/images`.


## Load data and configure style

The loader works when the notebook is run from the repository root or from `docs/`.


In [ ]:

from __future__ import annotations

import json
from collections import OrderedDict
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.patches import Patch

# ---------- Figure contract ----------
# Core conclusion: compare MATH training-stage changes from baseline to self-play,
# and compare PEFT-LoRA against Self-Play on the ScienceQA Nature Science subset.
# Archetype: quantitative grid; exports are PNG/PDF only.

PROJECT_CANDIDATES = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next((p.resolve() for p in PROJECT_CANDIDATES if (p / 'outputs' / 'eval').exists()), Path.cwd().resolve())
EVAL_DIR = PROJECT_ROOT / 'outputs' / 'eval'
IMAGE_DIR = PROJECT_ROOT / 'docs' / 'images'
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

MATH_STAGES = OrderedDict([
    ('baseline', 'Baseline'),
    ('sft', 'SFT'),
    ('rsft', 'R-SFT'),
    ('dpo', 'DPO'),
    ('grpo', 'GRPO'),
    ('self_play', 'Self-Play'),
])
SCIENCE_MODELS = OrderedDict([
    ('self_play', 'Self-Play'),
    ('peft_lora', 'PEFT-LoRA'),
])

MODEL_COLORS = {
    'baseline': '#6E6E6E',
    'sft': '#4C78A8',
    'rsft': '#59A14F',
    'dpo': '#B07AA1',
    'grpo': '#F28E2B',
    'self_play': '#2E6FBB',
    'peft_lora': '#C44E52',
}
METRIC_COLORS = {
    'accuracy': '#2E6FBB',
    'format_accuracy': '#C44E52',
}
HATCHES = {'accuracy': '///', 'format_accuracy': '\\\\'}

mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans', 'sans-serif'],
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'font.size': 7,
    'axes.spines.right': False,
    'axes.spines.top': False,
    'axes.linewidth': 0.8,
    'xtick.major.width': 0.7,
    'ytick.major.width': 0.7,
    'legend.frameon': False,
    'figure.dpi': 150,
    'savefig.dpi': 450,
})
sns.set_theme(context='paper', style='ticks', rc={
    'axes.facecolor': 'white',
    'figure.facecolor': 'white',
    'grid.color': '#E8E8E8',
})


def load_json(path: Path):
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)


def pct(x):
    return 100.0 * float(x)


def pct_fmt(x, pos=None):
    return f'{x:.0f}%'



def save_pub(fig: mpl.figure.Figure, stem: str, dpi: int = 450):
    saved = []
    for ext in ['png', 'pdf']:
        out = IMAGE_DIR / f'{stem}.{ext}'
        fig.savefig(out, bbox_inches='tight', dpi=dpi, facecolor='white')
        saved.append(out.name)
    print(f'Saved {stem}: ' + ', '.join(saved))


def load_math_frames():
    rows = []
    level_rows = []
    type_rows = []
    for stage, label in MATH_STAGES.items():
        summary = load_json(EVAL_DIR / stage / 'summary.json')
        rows.append({
            'stage': stage,
            'stage_label': label,
            'accuracy': pct(summary['accuracy']),
            'format_accuracy': pct(summary['format_accuracy']),
            'correct': summary['correct'],
            'num_examples': summary['num_examples'],
            'avg_response_tokens': summary.get('avg_response_tokens', np.nan),
            'elapsed_seconds': summary.get('generation_elapsed_seconds', np.nan),
        })
        for level, metrics in load_json(EVAL_DIR / stage / 'metrics_by_level.json').items():
            level_rows.append({
                'stage': stage,
                'stage_label': label,
                'level': level,
                'accuracy': pct(metrics['accuracy']),
                'format_accuracy': pct(metrics['format_accuracy']),
            })
        for typ, metrics in load_json(EVAL_DIR / stage / 'metrics_by_type.json').items():
            type_rows.append({
                'stage': stage,
                'stage_label': label,
                'type': typ,
                'accuracy': pct(metrics['accuracy']),
                'format_accuracy': pct(metrics['format_accuracy']),
            })
    return pd.DataFrame(rows), pd.DataFrame(level_rows), pd.DataFrame(type_rows)


def load_science_frames():
    summary = load_json(EVAL_DIR / 'peft' / 'summary.json')
    overall = []
    topic_rows = []
    category_rows = []
    for model, label in SCIENCE_MODELS.items():
        ms = summary['models'][model]
        overall.append({
            'model': model,
            'model_label': label,
            'answer_accuracy': pct(ms['accuracy']),
            'format_accuracy': pct(ms['format_accuracy']),
            'correct': ms['correct'],
            'num_examples': ms['num_examples'],
        })
        for topic, metrics in load_json(EVAL_DIR / 'peft' / f'metrics_by_topic_{model}.json').items():
            topic_rows.append({
                'model': model,
                'model_label': label,
                'topic': topic,
                'answer_accuracy': pct(metrics['accuracy']),
            })
        for cat, metrics in load_json(EVAL_DIR / 'peft' / f'metrics_by_category_{model}.json').items():
            category_rows.append({
                'model': model,
                'model_label': label,
                'category': cat,
                'answer_accuracy': pct(metrics['accuracy']),
            })
    return pd.DataFrame(overall), pd.DataFrame(topic_rows), pd.DataFrame(category_rows)


math_summary, math_level, math_type = load_math_frames()
science_overall, science_topic, science_category = load_science_frames()

print(f'Project root: {PROJECT_ROOT}')
print(f'Image output: {IMAGE_DIR}')
print(math_summary[['stage_label', 'accuracy', 'format_accuracy', 'avg_response_tokens']].to_string(index=False, float_format=lambda x: f'{x:.2f}'))
print(science_overall[['model_label', 'answer_accuracy', 'format_accuracy']].to_string(index=False, float_format=lambda x: f'{x:.2f}'))


## Generate and export figures

Each helper saves PNG and PDF under `docs/images/`. Bar charts use hatches for grayscale readability.


In [ ]:

def plot_math_overall_metrics():
    metrics = OrderedDict([
        ('accuracy', 'Answer Accuracy'),
        ('format_accuracy', 'Format Accuracy'),
    ])
    fig, ax = plt.subplots(figsize=(6.7, 3.25), constrained_layout=True)
    x = np.arange(len(math_summary))
    width = 0.34
    offsets = [-width / 2, width / 2]
    for i, (metric, label) in enumerate(metrics.items()):
        values = math_summary[metric].to_numpy()
        bars = ax.bar(
            x + offsets[i], values, width=width, label=label,
            color=METRIC_COLORS[metric], edgecolor='#2B2B2B', linewidth=0.45,
            hatch=HATCHES[metric], alpha=0.92,
        )
        for bar, value in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width() / 2, value + 1.3, f'{value:.2f}',
                    ha='center', va='bottom', fontsize=6.2)
    ax.set_xticks(x, math_summary['stage_label'])
    ax.set_ylabel('Score (%)')
    ax.set_ylim(0, 105)
    ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(pct_fmt))
    ax.grid(axis='y', linestyle='-', linewidth=0.35, alpha=0.58)
    ax.legend(ncols=2, loc='upper center', bbox_to_anchor=(0.5, -0.16), handlelength=1.8)
    ax.set_title('MATH Overall Metrics Across Training Stages', fontsize=8.8, fontweight='bold', loc='left', pad=8)
    sns.despine(ax=ax)
    save_pub(fig, 'math_overall_metrics')
    plt.show()



def plot_math_level_heatmap():
    pivot = math_level.pivot(index='stage_label', columns='level', values='accuracy').loc[list(MATH_STAGES.values())]
    fig, ax = plt.subplots(figsize=(6.2, 3.35), constrained_layout=True)
    cmap = sns.blend_palette(['#F7FAFF', '#C7D8EE', '#6E9ED0', '#1F5EA8'], as_cmap=True)
    sns.heatmap(
        pivot, ax=ax, cmap=cmap,
        vmin=0, vmax=100, annot=True, fmt='.2f',
        linewidths=1.0, linecolor='#F7F7F7', cbar_kws={'label': 'Accuracy (%)', 'shrink': 0.78, 'pad': 0.025, 'aspect': 24},
        annot_kws={'fontsize': 6.2},
    )
    ax.set_xlabel('MATH Difficulty Level')
    ax.set_ylabel('Training Stage')
    ax.set_title('MATH Accuracy by Difficulty Level', fontsize=8.8, fontweight='bold', loc='left', pad=8)
    save_pub(fig, 'math_level_heatmap')
    plt.show()


def plot_math_type_heatmap():
    pivot = math_type.pivot(index='stage_label', columns='type', values='accuracy').loc[list(MATH_STAGES.values())]
    ordered_cols = pivot.loc['Self-Play'].sort_values(ascending=False).index.tolist()
    pivot = pivot[ordered_cols]
    fig, ax = plt.subplots(figsize=(10.4, 3.75), constrained_layout=True)
    cmap = sns.blend_palette(['#FFF9F5', '#E9C5C0', '#C86E72', '#8F2F4A'], as_cmap=True)
    sns.heatmap(
        pivot, ax=ax, cmap=cmap,
        vmin=0, vmax=100, annot=True, fmt='.2f',
        linewidths=1.0, linecolor='#F7F7F7', cbar_kws={'label': 'Accuracy (%)', 'shrink': 0.76, 'pad': 0.025, 'aspect': 24},
        annot_kws={'fontsize': 7.0},
    )
    ax.set_xlabel('MATH Problem Type')
    ax.set_ylabel('Training Stage')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0, ha='center', fontsize=5.8)
    ax.set_title('MATH Accuracy by Problem Type', fontsize=8.8, fontweight='bold', loc='left', pad=8)
    save_pub(fig, 'math_type_heatmap')
    plt.show()


def plot_scienceqa_overall_metrics():
    metrics = OrderedDict([
        ('answer_accuracy', 'Answer Accuracy'),
        ('format_accuracy', 'Format Accuracy'),
    ])
    fig, ax = plt.subplots(figsize=(4.2, 3.25), constrained_layout=True)
    x = np.arange(len(science_overall))
    width = 0.34
    offsets = [-width / 2, width / 2]
    for i, (metric, label) in enumerate(metrics.items()):
        color_key = 'accuracy' if metric == 'answer_accuracy' else 'format_accuracy'
        values = science_overall[metric].to_numpy()
        bars = ax.bar(
            x + offsets[i], values, width=width, label=label,
            color=METRIC_COLORS[color_key], edgecolor='#2B2B2B', linewidth=0.45,
            hatch=HATCHES[color_key], alpha=0.92,
        )
        for bar, value in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width() / 2, value + 1.1, f'{value:.2f}',
                    ha='center', va='bottom', fontsize=6.3)
    ax.set_xticks(x, science_overall['model_label'])
    ax.set_ylabel('Score (%)')
    ax.set_ylim(0, 106)
    ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(pct_fmt))
    ax.grid(axis='y', linewidth=0.35, alpha=0.55)
    ax.legend(ncols=2, loc='upper center', bbox_to_anchor=(0.5, -0.16), handlelength=1.8, columnspacing=1.2)
    ax.set_title('ScienceQA Nature Science Subset Overall Metrics', fontsize=8.8, fontweight='bold', loc='left', pad=8)
    sns.despine(ax=ax)
    save_pub(fig, 'scienceqa_overall_metrics')
    plt.show()


def plot_scienceqa_category_accuracy():
    pivot = science_category.pivot(index='category', columns='model_label', values='answer_accuracy')
    pivot = pivot.assign(delta=pivot['PEFT-LoRA'] - pivot['Self-Play']).sort_values('delta', ascending=False)
    pivot = pivot[['Self-Play', 'PEFT-LoRA']]
    fig, ax = plt.subplots(figsize=(8.6, 7.35), constrained_layout=True)
    cmap = sns.blend_palette(['#FBF8FF', '#D9CBE8', '#9A79BE', '#5C3F8F'], as_cmap=True)
    sns.heatmap(
        pivot, ax=ax, cmap=cmap,
        vmin=0, vmax=100, annot=True, fmt='.2f', linewidths=0.85, linecolor='#F7F7F7',
        cbar_kws={'label': 'Answer Accuracy (%)', 'shrink': 0.70, 'pad': 0.025, 'aspect': 28}, annot_kws={'fontsize': 5.6},
    )
    ax.set_xlabel('Model')
    ax.set_ylabel('ScienceQA Category')
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=5.7)
    ax.set_title('ScienceQA Nature Science Subset Category Answer Accuracy', fontsize=8.8, fontweight='bold', loc='left', pad=8)
    save_pub(fig, 'scienceqa_category_accuracy')
    plt.show()


def plot_scienceqa_topic_accuracy():
    pivot = science_topic.pivot(index='topic', columns='model_label', values='answer_accuracy')
    pivot = pivot.assign(delta=pivot['PEFT-LoRA'] - pivot['Self-Play']).sort_values('delta', ascending=False)
    pivot = pivot[['Self-Play', 'PEFT-LoRA']]
    fig, ax = plt.subplots(figsize=(7.0, 3.05), constrained_layout=True)
    cmap = sns.blend_palette(['#F6FCF9', '#C6DED4', '#78AD97', '#347A63'], as_cmap=True)
    sns.heatmap(
        pivot, ax=ax, cmap=cmap,
        vmin=0, vmax=100, annot=True, fmt='.2f', linewidths=0.9, linecolor='#F7F7F7',
        cbar_kws={'label': 'Answer Accuracy (%)', 'shrink': 0.76, 'pad': 0.025, 'aspect': 22}, annot_kws={'fontsize': 6.2},
    )
    ax.set_xlabel('Model')
    ax.set_ylabel('ScienceQA Topic')
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=6.2)
    ax.set_title('ScienceQA Nature Science Subset Topic Answer Accuracy', fontsize=8.8, fontweight='bold', loc='left', pad=8)
    save_pub(fig, 'scienceqa_topic_accuracy')
    plt.show()


plot_math_overall_metrics()
plot_math_level_heatmap()
plot_math_type_heatmap()
plot_scienceqa_overall_metrics()
plot_scienceqa_category_accuracy()
plot_scienceqa_topic_accuracy()
